# California Housing Price Prediction: Exploratory & Model Diagnostics

Self-contained notebook for Google Colab or local Jupyter. It loads the 1990 U.S. Census California Housing dataset, inspects target censoring at `$500,001`, compares Ridge Regression, Random Forest, and a tuned MLP Neural Network, and computes hold-out residual diagnostics and permutation importance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df["target"] = housing.target

capped = (df["target"] >= 5.00001)
print(f"Rows: {len(df):,} | Features: {len(housing.feature_names)} | Nulls: {int(df.isnull().sum().sum())}")
print(f"Right-censored at $500,001: {int(capped.sum()):,} ({capped.mean()*100:.2f}%)")
df.describe().T[["mean", "std", "min", "50%", "max"]]

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate(name, model):
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    return {
        "Model": name,
        "R2": round(r2_score(y_test, preds), 4),
        "RMSE": round(float(np.sqrt(mse)), 4),
        "MAE": round(mean_absolute_error(y_test, preds), 4),
        "MSE": round(mse, 4),
    }

ridge = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))]).fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42).fit(X_train, y_train)

mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(max_iter=500, early_stopping=True, random_state=42)),
])
search = RandomizedSearchCV(
    mlp_pipe,
    {
        "mlp__hidden_layer_sizes": [(64, 32), (128, 64, 32), (256, 128, 64), (32, 16)],
        "mlp__activation": ["relu", "tanh"],
        "mlp__alpha": [0.0001, 0.001, 0.01],
        "mlp__learning_rate_init": [0.001, 0.005, 0.01],
    },
    n_iter=12,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
).fit(X_train, y_train)
mlp_best = search.best_estimator_

results = pd.DataFrame([
    evaluate("Ridge Regression", ridge),
    evaluate("Random Forest", rf),
    evaluate("MLP Neural Network", mlp_best),
]).sort_values("R2", ascending=False)
print("Best MLP params:", search.best_params_)
results

In [ ]:
preds = mlp_best.predict(X_test)
residuals = y_test - preds
capped_test = y_test >= 5.00001

print(f"Mean residual bias: {residuals.mean():+.4f}")
print(f"Uncensored RMSE: {np.sqrt(mean_squared_error(y_test[~capped_test], preds[~capped_test])):.4f}")
print(f"Censored RMSE:   {np.sqrt(mean_squared_error(y_test[capped_test], preds[capped_test])):.4f}")

perm = permutation_importance(mlp_best, X_test, y_test, n_repeats=5, random_state=42, scoring="r2")
imp_df = pd.DataFrame({
    "Feature": housing.feature_names,
    "Mean_Delta_R2": perm.importances_mean,
    "Std": perm.importances_std,
}).sort_values("Mean_Delta_R2", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(preds, residuals, alpha=0.25, s=10)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("MLP Hold-Out Residuals (y - y_hat)")
axes[0].set_xlabel("Predicted ($100k)")
axes[0].set_ylabel("Residual ($100k)")

axes[1].barh(imp_df["Feature"], imp_df["Mean_Delta_R2"], xerr=imp_df["Std"])
axes[1].set_title("Permutation Importance (Hold-Out R2 Drop)")
axes[1].set_xlabel("Mean Delta R2")
plt.tight_layout()
plt.show()